In [ ]:
import pandas as pd
from scipy.stats import fisher_exact
import numpy as np
from scipy.stats import wilcoxon


In [ ]:
def get_motif_wise_densities(fimo_tsv, fasta_file, num_motifs):
    df = pd.read_csv(fimo_tsv, sep='\t', skipfooter=3, engine='python')
    
    from Bio import SeqIO
    total_kb = sum(len(record.seq) for record in SeqIO.parse(fasta_file, "fasta")) / 1000
    
    motif_counts = df.groupby('motif_id').size()
    
    densities = (motif_counts / total_kb).to_dict()
    
    return densities

In [ ]:

import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from Bio import SeqIO

def get_motif_sequence_fraction(fimo_tsv, fasta_file):
    total_seqs = sum(1 for _ in SeqIO.parse(fasta_file, "fasta"))
    
    df = pd.read_csv(fimo_tsv, sep='\t', skipfooter=3, engine='python')
    
    unique_hits = df.groupby('motif_id')['sequence_name'].nunique()
    
    fractions = (unique_hits / total_seqs).to_dict()
    
    return fractions

# ==========================================
# ==========================================
ev_motifs_on_cyto = get_motif_sequence_fraction("fimo_EV_motif_on_Cyto_seq/fimo.tsv", "../circRNA_ligated/Cyto_sequences.fasta")
ev_motifs_on_ev = get_motif_sequence_fraction("fimo_EV_motif_on_EV_seq/fimo.tsv", "../circRNA_ligated/EV_sequences.fasta")

all_ev_motif_ids = set(ev_motifs_on_ev.keys()) | set(ev_motifs_on_cyto.keys())
pair_ev_EVseq = [ev_motifs_on_ev.get(m, 0) for m in all_ev_motif_ids]
pair_ev_Cytoseq = [ev_motifs_on_cyto.get(m, 0) for m in all_ev_motif_ids]

stat_ev, pval_ev = wilcoxon(pair_ev_EVseq, pair_ev_Cytoseq, alternative='greater')

print("=== EV Motifs enrichment analysis (sequence proportion) ===")
print(f"On average, {np.mean(pair_ev_EVseq)*100:.2f}% of sequences contain these motifs")
print(f"On average, {np.mean(pair_ev_Cytoseq)*100:.2f}% of sequences contain these motifs")
print(f"Paired-test P-value: {pval_ev:.4e}\n")


# ==========================================
# ==========================================
cyto_motifs_on_cyto = get_motif_sequence_fraction("fimo_Cyto_motif_on_Cyto_seq/fimo.tsv", "../circRNA_ligated/Cyto_sequences.fasta")
cyto_motifs_on_ev = get_motif_sequence_fraction("fimo_Cyto_motif_on_EV_seq/fimo.tsv", "../circRNA_ligated/EV_sequences.fasta")

all_cyto_motif_ids = set(cyto_motifs_on_cyto.keys()) | set(cyto_motifs_on_ev.keys())
pair_cyto_Cytoseq = [cyto_motifs_on_cyto.get(m, 0) for m in all_cyto_motif_ids]
pair_cyto_EVseq = [cyto_motifs_on_ev.get(m, 0) for m in all_cyto_motif_ids]

stat_cyto, pval_cyto = wilcoxon(pair_cyto_Cytoseq, pair_cyto_EVseq, alternative='greater')

print("=== Cyto Motifs enrichment analysis (sequence proportion) ===")
print(f"On average, {np.mean(pair_cyto_Cytoseq)*100:.2f}% of sequences contain these motifs")
print(f"On average, {np.mean(pair_cyto_EVseq)*100:.2f}% of sequences contain these motifs")
print(f"Paired-test P-value: {pval_cyto:.4e}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

data = []

for val in pair_ev_EVseq:
    data.append({'Sequence Fraction (%)': val * 100, 'Motif Type': 'EV-motifs', 'Sequence Background': 'EV sequences'})
for val in pair_ev_Cytoseq:
    data.append({'Sequence Fraction (%)': val * 100, 'Motif Type': 'EV-motifs', 'Sequence Background': 'Cellular sequences'})

for val in pair_cyto_EVseq:
    data.append({'Sequence Fraction (%)': val * 100, 'Motif Type': 'Cellular-motifs', 'Sequence Background': 'EV sequences'})
for val in pair_cyto_Cytoseq:
    data.append({'Sequence Fraction (%)': val * 100, 'Motif Type': 'Cellular-motifs', 'Sequence Background': 'Cellular sequences'})

df_plot = pd.DataFrame(data)

def get_asterisks(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    return 'ns'
# ==========================================================

plt.figure(figsize=(8, 6))
sns.set_theme(style="ticks", font_scale=1.1)

palette = {'EV sequences': '#98B2CD', 'Cellular sequences': '#D89F7B'}

ax = sns.boxplot(
    data=df_plot, 
    x='Motif Type', 
    y='Sequence Fraction (%)', 
    hue='Sequence Background',
    palette=palette,
    width=0.5,
    showfliers=True, 
    linewidth=1.5
)

y_range = df_plot['Sequence Fraction (%)'].max() - df_plot['Sequence Fraction (%)'].min()
y_padding = y_range * 0.05  
h = y_range * 0.015

max_ev = df_plot[df_plot['Motif Type'] == 'EV-motifs']['Sequence Fraction (%)'].max()
max_cyto = df_plot[df_plot['Motif Type'] == 'Cellular-motifs']['Sequence Fraction (%)'].max()

x1_ev, x2_ev = 0 - 0.125, 0 + 0.125
y_ev = max_ev + y_padding
ax.plot([x1_ev, x1_ev, x2_ev, x2_ev], [y_ev, y_ev+h, y_ev+h, y_ev], lw=1.2, color='black')
ax.text((x1_ev+x2_ev)/2, y_ev+h, get_asterisks(pval_ev), ha='center', va='bottom', color='black', fontsize=12, fontweight='bold')

x1_cyto, x2_cyto = 1 - 0.125, 1 + 0.125
y_cyto = max_cyto + y_padding
ax.plot([x1_cyto, x1_cyto, x2_cyto, x2_cyto], [y_cyto, y_cyto+h, y_cyto+h, y_cyto], lw=1.2, color='black')
ax.text((x1_cyto+x2_cyto)/2, y_cyto+h, get_asterisks(pval_cyto), ha='center', va='bottom', color='black', fontsize=12, fontweight='bold')
# ==========================================================

current_ymin, current_ymax = ax.get_ylim()
ax.set_ylim(current_ymin, max(current_ymax, max(y_ev, y_cyto) + y_padding * 4.5))

plt.title('Motif Sequence Fraction Enrichment', fontsize=14, pad=15, fontweight='bold')
plt.ylabel('Percentage of Sequences with Motif (%)', fontsize=13)
plt.xlabel('')

plt.legend(title='Sequence Background', loc=0, bbox_to_anchor=(1, 1), framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from Bio import SeqIO

def get_sequence_level_densities(fimo_tsv, fasta_file):
    """Calculate total target-motif hit density for each sequence (hits/kb)."""
    seq_lengths = {}
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq_lengths[record.id] = len(record.seq) / 1000.0
        
    df = pd.read_csv(fimo_tsv, sep='\t', skipfooter=3, engine='python')
    
    hits_per_seq = df.groupby('sequence_name').size()
    
    seq_densities = []
    for seq_id, length_kb in seq_lengths.items():
        hits = hits_per_seq.get(seq_id, 0)
        seq_densities.append(hits / length_kb)
        
    return seq_densities

# ==========================================
# ==========================================
ev_seq_densities_EV_motifs = get_sequence_level_densities(
    "fimo_EV_motif_on_EV_seq/fimo.tsv", 
    "../circRNA_ligated/EV_sequences.fasta"
)
cyto_seq_densities_EV_motifs = get_sequence_level_densities(
    "fimo_EV_motif_on_Cyto_seq/fimo.tsv", 
    "../circRNA_ligated/Cyto_sequences.fasta"
)

stat_ev, pval_ev = mannwhitneyu(ev_seq_densities_EV_motifs, cyto_seq_densities_EV_motifs, alternative='greater')

print("=== EV Motifs enrichment analysis (sequence-level motif frequency) ===")
print(f"Number of EV sequences: {len(ev_seq_densities_EV_motifs)}, mean frequency: {np.mean(ev_seq_densities_EV_motifs):.4f} hits/kb")
print(f"Number of Cyto sequences: {len(cyto_seq_densities_EV_motifs)}, mean frequency: {np.mean(cyto_seq_densities_EV_motifs):.4f} hits/kb")
print(f"Mann-Whitney U-test P-value: {pval_ev:.4e}\n")


# ==========================================
# ==========================================
cyto_seq_densities_Cyto_motifs = get_sequence_level_densities(
    "fimo_Cyto_motif_on_Cyto_seq/fimo.tsv", 
    "../circRNA_ligated/Cyto_sequences.fasta"
)
ev_seq_densities_Cyto_motifs = get_sequence_level_densities(
    "fimo_Cyto_motif_on_EV_seq/fimo.tsv", 
    "../circRNA_ligated/EV_sequences.fasta"
)

stat_cyto, pval_cyto = mannwhitneyu(cyto_seq_densities_Cyto_motifs, ev_seq_densities_Cyto_motifs, alternative='greater')

print("=== Cyto Motifs enrichment analysis (sequence-level motif frequency) ===")
print(f"Number of Cyto sequences: {len(cyto_seq_densities_Cyto_motifs)}, mean frequency: {np.mean(cyto_seq_densities_Cyto_motifs):.4f} hits/kb")
print(f"Number of EV sequences: {len(ev_seq_densities_Cyto_motifs)}, mean frequency: {np.mean(ev_seq_densities_Cyto_motifs):.4f} hits/kb")
print(f"Mann-Whitney U-test P-value: {pval_cyto:.4e}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Build the plotting DataFrame.
data = []

for val in ev_seq_densities_EV_motifs:
    data.append({'Density (hits/kb)': val, 'Motif Type': 'EV-motifs', 'Sequence Background': 'EV sequences'})
for val in cyto_seq_densities_EV_motifs:
    data.append({'Density (hits/kb)': val, 'Motif Type': 'EV-motifs', 'Sequence Background': 'Cellular sequences'})

for val in ev_seq_densities_Cyto_motifs:
    data.append({'Density (hits/kb)': val, 'Motif Type': 'Cellular-motifs', 'Sequence Background': 'EV sequences'})
for val in cyto_seq_densities_Cyto_motifs:
    data.append({'Density (hits/kb)': val, 'Motif Type': 'Cellular-motifs', 'Sequence Background': 'Cellular sequences'})

df_plot = pd.DataFrame(data)

def get_asterisks(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    return 'ns'
# ==========================================================

plt.figure(figsize=(8, 6))
sns.set_theme(style="ticks", font_scale=1.1)
palette = {'EV sequences': '#98B2CD', 'Cellular sequences': '#D89F7B'}

ax = sns.boxplot(
    data=df_plot, 
    x='Motif Type', 
    y='Density (hits/kb)', 
    hue='Sequence Background',
    palette=palette,
    width=0.5,
    showfliers=False,
    linewidth=1.5
)

current_ymin, current_ymax = ax.get_ylim()
y_range = current_ymax - current_ymin
y_padding = y_range * 0.05
h = y_range * 0.02

y_bracket = current_ymax + y_padding

x1_ev, x2_ev = 0 - 0.125, 0 + 0.125
ax.plot([x1_ev, x1_ev, x2_ev, x2_ev], [y_bracket, y_bracket+h, y_bracket+h, y_bracket], lw=1.2, color='black')
ax.text((x1_ev+x2_ev)/2, y_bracket+h, get_asterisks(pval_ev), ha='center', va='bottom', color='black', fontsize=12, fontweight='bold')

x1_cyto, x2_cyto = 1 - 0.125, 1 + 0.125
ax.plot([x1_cyto, x1_cyto, x2_cyto, x2_cyto], [y_bracket, y_bracket+h, y_bracket+h, y_bracket], lw=1.2, color='black')
ax.text((x1_cyto+x2_cyto)/2, y_bracket+h, get_asterisks(pval_cyto), ha='center', va='bottom', color='black', fontsize=12, fontweight='bold')
# ==========================================================

ax.set_ylim(current_ymin, y_bracket + h + y_padding * 3.5)

plt.title('Sequence-Level Motif Density Enrichment', fontsize=14, pad=15, fontweight='bold')
plt.ylabel('Motif Frequency (hits / kb)', fontsize=13)
plt.xlabel('')

plt.legend(title='Sequence Background', bbox_to_anchor=(1.05, 1), loc='upper left', framealpha=0.9)


plt.tight_layout()
plt.show()